In [2]:
import pyodbc
import pygrametl

# Update these with your actual SSMS details
STAGING_CONNECTION_STRING = (
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=DESKTOP-AR905LJ;"
    "Database=staging;"
    "UID=sa;"
    "PWD=ghom3220;"
)
DWH_CONNECTION_STRING = ( 
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=DESKTOP-AR905LJ;"
    "Database=DWH;"
    "UID=sa;"
    "PWD=ghom3220;"
)

In [3]:
source_conn = pyodbc.connect(STAGING_CONNECTION_STRING)
source_connection = pygrametl.ConnectionWrapper(source_conn)
source_cursor = source_connection.cursor()
source_cursor.execute("SELECT @@version;")
result = source_cursor.fetchone()
print(f"connected to {result[0]}!")

connected to Microsoft SQL Server 2025 (RTM) - 17.0.1000.7 (X64) 
	Oct 21 2025 12:05:57 
	Copyright (C) 2025 Microsoft Corporation
	Standard Developer Edition (64-bit) on Windows 10 Pro 10.0 <X64> (Build 26200: ) (Hypervisor)
!


In [4]:
dwh_conn = pyodbc.connect(DWH_CONNECTION_STRING)
dwh_connection = pygrametl.ConnectionWrapper(dwh_conn)
dwh_cursor = dwh_connection.cursor()
dwh_cursor.execute("SELECT @@version;")
result = dwh_cursor.fetchone()
print(f"connected to {result[0]}!")
dwh_connection.setasdefault()

connected to Microsoft SQL Server 2025 (RTM) - 17.0.1000.7 (X64) 
	Oct 21 2025 12:05:57 
	Copyright (C) 2025 Microsoft Corporation
	Standard Developer Edition (64-bit) on Windows 10 Pro 10.0 <X64> (Build 26200: ) (Hypervisor)
!


In [5]:
create_table_sql = """
IF NOT EXISTS (SELECT * FROM sys.objects WHERE object_id = OBJECT_ID(N'[dbo].[Dim_Channel]') AND type in (N'U'))
BEGIN
    CREATE TABLE Dim_Channel (
        channel_ID VARCHAR(50) PRIMARY KEY,
        channel_name VARCHAR(100),
    );
END
"""
dwh_cursor.execute(create_table_sql)
dwh_connection.commit()

In [6]:
from pygrametl.tables import Dimension
Dim_Channel = pygrametl.tables.Dimension(
    name='Dim_Channel',

    key='channel_ID',

    attributes=[
        'channel_name'
    ]
)    

In [7]:
channel_source=source_cursor.execute("SELECT Distinct CanalVente FROM source")
for row in channel_source:
    channel_data = {
        "channel_name" : row.CanalVente
    }
    print(channel_data)
    Dim_Channel.ensure(channel_data)
dwh_connection.commit()
dwh_cursor.close()
dwh_connection.close()

{'channel_name': 'Site web'}
{'channel_name': 'Magasin physique'}
{'channel_name': 'Application mobile'}
{'channel_name': 'Réseaux sociaux'}
{'channel_name': 'Téléphone'}
